# 95. The Supply Chain Network Design under Uncertainty Problem
## Tier 1: Mathematical Formulation (Integer Programming)

### Key assumptions
- Multiple uncertainty scenarios with known probabilities
- Binary facility location decisions consistent across scenarios
- Customer assignments can vary by scenario
- Fixed facility costs and variable operational costs
- Capacity constraints at each facility

### Approach (step-by-step)
1. **Problem Definition**: Define sets for facilities, customers, products, and scenarios
2. **Decision Variables**: Binary variables for facility opening, continuous variables for shipments
3. **Objective Function**: Minimize total expected cost (fixed + expected variable)
4. **Constraints**: Demand satisfaction, capacity limits, assignment consistency
5. **Solution Method**: Use pulp solver for exact optimization

### What to look for in the results
- Which facilities should be opened (binary decisions)
- How customers are assigned in each scenario
- Total expected cost breakdown
- Capacity utilization across scenarios

### Concrete example (from the source)
Three-facility, four-customer network with two demand scenarios:
- Fixed costs: [100, 120, 90]
- High demand: [50, 40, 35, 45]
- Low demand: [30, 25, 20, 25]
- Capacities: [80, 90, 70]
- Scenario probabilities: 0.5 each

### Visualization(s)
- Network structure with facility-customer assignments
- Cost breakdown analysis
- Capacity utilization by scenario

### Why this Tier exists vs earlier Tiers
This is Tier 1, providing the mathematical foundation with exact optimization and provable optimality.

### Pros / Cons vs other approaches
**Pros:**
- Guaranteed optimal solution
- Clear mathematical formulation
- Handles uncertainty explicitly through scenarios

**Cons:**
- Computationally intensive for large instances
- Requires discrete scenario representation
- May not scale to hundreds of facilities

### When to use this Tier
- Small to medium-sized networks (≤20 facilities)
- When optimality guarantees are required
- Strategic planning with limited uncertainty scenarios

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pulp import *
import networkx as nx
from dataclasses import dataclass
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class SupplyChainInstance:
    """Data structure for supply chain network design instance"""
    facilities: List[int]  # Facility indices
    customers: List[int]   # Customer indices  
    scenarios: List[int]   # Scenario indices
    fixed_costs: List[float]  # Fixed costs for opening facilities
    demands: Dict[Tuple[int, int, int], float]  # (customer, product, scenario) -> demand
    capacities: List[float]  # Facility capacities
    transport_costs: Dict[Tuple[int, int], float]  # (facility, customer) -> unit cost
    scenario_probs: List[float]  # Scenario probabilities

def create_example_instance():
    """Create the concrete example from the source material"""
    facilities = [0, 1, 2]  # 3 facilities
    customers = [0, 1, 2, 3]  # 4 customers
    scenarios = [0, 1]  # 2 scenarios (high/low demand)
    
    # Fixed costs for facilities
    fixed_costs = [100, 120, 90]
    
    # Capacities
    capacities = [80, 90, 70]
    
    # Scenario probabilities
    scenario_probs = [0.5, 0.5]
    
    # Transportation costs (facility, customer)
    transport_costs = {
        (0, 0): 2.5, (0, 1): 3.0, (0, 2): 3.5, (0, 3): 4.0,
        (1, 0): 3.2, (1, 1): 2.1, (1, 2): 4.1, (1, 3): 3.6,
        (2, 0): 4.1, (2, 1): 4.6, (2, 2): 2.0, (2, 3): 2.4
    }
    
    # Demands (customer, scenario) -> demand
    demands = {}
    # High demand scenario (0)
    high_demand = [50, 40, 35, 45]
    for c, d in enumerate(high_demand):
        demands[(c, 0, 0)] = d  # customer, product, scenario
    
    # Low demand scenario (1)
    low_demand = [30, 25, 20, 25]
    for c, d in enumerate(low_demand):
        demands[(c, 0, 1)] = d
    
    return SupplyChainInstance(
        facilities=facilities,
        customers=customers,
        scenarios=scenarios,
        fixed_costs=fixed_costs,
        demands=demands,
        capacities=capacities,
        transport_costs=transport_costs,
        scenario_probs=scenario_probs
    )

# Create the instance
instance = create_example_instance()
print(f"Facilities: {instance.facilities}")
print(f"Customers: {instance.customers}")
print(f"Scenarios: {instance.scenarios}")
print(f"Fixed costs: {instance.fixed_costs}")
print(f"Capacities: {instance.capacities}")

In [ ]:
def solve_supply_chain_network_design(instance: SupplyChainInstance):
    """Solve the supply chain network design problem using integer programming"""
    
    # Create the optimization problem
    prob = LpProblem("Supply_Chain_Network_Design", LpMinimize)
    
    # Decision variables
    # y_i: binary variable for opening facility i
    y = {}
    for i in instance.facilities:
        y[i] = LpVariable(f"y_{i}", cat='Binary')
    
    # x_{ijks}: continuous variable for shipments
    x = {}
    for i in instance.facilities:
        for j in instance.customers:
            for k in [0]:  # Single product
                for s in instance.scenarios:
                    x[(i, j, k, s)] = LpVariable(f"x_{i}_{j}_{k}_{s}", lowBound=0)
    
    # z_{ijs}: binary variable for customer assignment
    z = {}
    for i in instance.facilities:
        for j in instance.customers:
            for s in instance.scenarios:
                z[(i, j, s)] = LpVariable(f"z_{i}_{j}_{s}", cat='Binary')
    
    # Objective function: minimize total expected cost
    # Fixed costs + expected variable costs
    fixed_cost_component = lpSum([instance.fixed_costs[i] * y[i] for i in instance.facilities])
    
    variable_cost_component = lpSum([
        instance.scenario_probs[s] * instance.transport_costs[(i, j)] * x[(i, j, 0, s)]
        for i in instance.facilities
        for j in instance.customers
        for s in instance.scenarios
    ])
    
    prob += fixed_cost_component + variable_cost_component, "Total_Expected_Cost"
    
    # Constraints
    
    # 1. Demand satisfaction constraint
    for j in instance.customers:
        for k in [0]:  # Single product
            for s in instance.scenarios:
                prob += lpSum([x[(i, j, k, s)] for i in instance.facilities]) == instance.demands[(j, k, s)], f"Demand_{j}_{k}_{s}"
    
    # 2. Facility capacity constraint
    for i in instance.facilities:
        for s in instance.scenarios:
            prob += lpSum([x[(i, j, 0, s)] for j in instance.customers]) <= instance.capacities[i] * y[i], f"Capacity_{i}_{s}"
    
    # 3. Service assignment constraint (link x and z)
    M = 1000  # Large constant
    for i in instance.facilities:
        for j in instance.customers:
            for k in [0]:
                for s in instance.scenarios:
                    prob += x[(i, j, k, s)] <= M * z[(i, j, s)], f"Assignment_{i}_{j}_{k}_{s}"
    
    # 4. Single assignment constraint
    for j in instance.customers:
        for s in instance.scenarios:
            prob += lpSum([z[(i, j, s)] for i in instance.facilities]) == 1, f"Single_Assignment_{j}_{s}"
    
    # 5. Facility operation constraint
    for i in instance.facilities:
        for j in instance.customers:
            for s in instance.scenarios:
                prob += z[(i, j, s)] <= y[i], f"Facility_Operation_{i}_{j}_{s}"
    
    # Solve the problem
    print("Solving supply chain network design problem...")
    prob.solve(PULP_CBC_CMD(msg=False))
    
    return prob, y, x, z

# Solve the problem
prob, y_vars, x_vars, z_vars = solve_supply_chain_network_design(instance)

print(f"\nProblem Status: {LpStatus[prob.status]}")
print(f"Optimal Objective Value: ${prob.objective.value:,.2f}")

In [ ]:
def extract_solution(instance, prob, y_vars, x_vars, z_vars):
    """Extract and format the solution"""
    
    if prob.status != 1:  # Not optimal
        return None
    
    # Extract facility opening decisions
    opened_facilities = []
    for i in instance.facilities:
        if y_vars[i].value() > 0.5:
            opened_facilities.append(i)
    
    # Extract customer assignments by scenario
    assignments = {}
    shipments = {}
    
    for s in instance.scenarios:
        assignments[s] = {}
        shipments[s] = {}
        
        for j in instance.customers:
            for i in instance.facilities:
                if z_vars[(i, j, s)].value() > 0.5:
                    assignments[s][j] = i
                    shipments[s][(i, j)] = x_vars[(i, j, 0, s)].value()
    
    # Calculate cost components
    fixed_cost = sum(instance.fixed_costs[i] for i in opened_facilities)
    
    variable_costs = {}
    total_variable_cost = 0
    
    for s in instance.scenarios:
        scenario_cost = 0
        for (i, j), quantity in shipments[s].items():
            scenario_cost += instance.transport_costs[(i, j)] * quantity
        variable_costs[s] = scenario_cost
        total_variable_cost += instance.scenario_probs[s] * scenario_cost
    
    total_cost = fixed_cost + total_variable_cost
    
    return {
        'opened_facilities': opened_facilities,
        'assignments': assignments,
        'shipments': shipments,
        'fixed_cost': fixed_cost,
        'variable_costs': variable_costs,
        'total_variable_cost': total_variable_cost,
        'total_cost': total_cost
    }

# Extract solution
solution = extract_solution(instance, prob, y_vars, x_vars, z_vars)

if solution:
    print("\n=== OPTIMAL SOLUTION ===")
    print(f"Opened facilities: {solution['opened_facilities']}")
    print(f"Fixed cost: ${solution['fixed_cost']:,.2f}")
    print(f"Total expected variable cost: ${solution['total_variable_cost']:,.2f}")
    print(f"Total expected cost: ${solution['total_cost']:,.2f}")
    
    print("\nCustomer assignments by scenario:")
    for s in instance.scenarios:
        scenario_name = "High Demand" if s == 0 else "Low Demand"
        print(f"\n{scenario_name} Scenario (p={instance.scenario_probs[s]}):")
        for j in instance.customers:
            facility = solution['assignments'][s][j]
            quantity = solution['shipments'][s][(facility, j)]
            cost = instance.transport_costs[(facility, j)] * quantity
            print(f"  Customer {j} -> Facility {facility}: {quantity:.0f} units, cost: ${cost:.2f}")
        
        print(f"  Scenario variable cost: ${solution['variable_costs'][s]:,.2f}")
else:
    print("No optimal solution found!")

In [ ]:
def visualize_network_solution(instance, solution):
    """Create visualizations for the network design solution"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Supply Chain Network Design - Mathematical Formulation Results', fontsize=16, fontweight='bold')
    
    # 1. Network structure for high demand scenario
    ax1 = axes[0, 0]
    G = nx.Graph()
    
    # Add nodes
    for i in instance.facilities:
        is_open = i in solution['opened_facilities']
        G.add_node(f"F{i}", type='facility', open=is_open)
    
    for j in instance.customers:
        G.add_node(f"C{j}", type='customer')
    
    # Add edges for high demand scenario
    for j in instance.customers:
        facility = solution['assignments'][0][j]  # High demand scenario
        G.add_edge(f"F{facility}", f"C{j}")
    
    # Position nodes
    pos = {}
    # Facilities on top
    for idx, i in enumerate(instance.facilities):
        pos[f"F{i}"] = (idx * 2, 2)
    # Customers on bottom
    for idx, j in enumerate(instance.customers):
        pos[f"C{j}"] = (idx * 2 + 1, 0)
    
    # Draw nodes
    facility_colors = ['green' if i in solution['opened_facilities'] else 'red' 
                      for i in instance.facilities]
    nx.draw_networkx_nodes(G, pos, nodelist=[f"F{i}" for i in instance.facilities],
                           node_color=facility_colors, node_size=800, ax=ax1)
    nx.draw_networkx_nodes(G, pos, nodelist=[f"C{j}" for j in instance.customers],
                           node_color='blue', node_size=600, ax=ax1)
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, edge_color='gray', width=2, ax=ax1)
    nx.draw_networkx_labels(G, pos, ax=ax1)
    ax1.set_title('Network Structure (High Demand Scenario)')
    ax1.axis('off')
    
    # 2. Cost breakdown
    ax2 = axes[0, 1]
    cost_components = ['Fixed Cost', 'Expected Variable Cost']
    cost_values = [solution['fixed_cost'], solution['total_variable_cost']]
    colors = ['#FF6B6B', '#4ECDC4']
    
    bars = ax2.bar(cost_components, cost_values, color=colors)
    ax2.set_ylabel('Cost ($)')
    ax2.set_title('Cost Breakdown')
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, cost_values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(cost_values)*0.01,
                f'${value:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    # 3. Capacity utilization by scenario
    ax3 = axes[1, 0]
    
    # Calculate capacity utilization
    utilization_data = []
    for s in instance.scenarios:
        scenario_name = "High Demand" if s == 0 else "Low Demand"
        for i in solution['opened_facilities']:
            total_shipment = sum(solution['shipments'][s].get((i, j), 0) for j in instance.customers)
            utilization = total_shipment / instance.capacities[i] * 100
            utilization_data.append({'Facility': f'F{i}', 'Scenario': scenario_name, 'Utilization': utilization})
    
    utilization_df = pd.DataFrame(utilization_data)
    
    # Grouped bar chart
    facilities = list(set(utilization_df['Facility']))
    scenarios = ['High Demand', 'Low Demand']
    
    x = np.arange(len(facilities))
    width = 0.35
    
    high_util = [utilization_df[(utilization_df['Facility'] == f) & (utilization_df['Scenario'] == 'High Demand')]['Utilization'].values[0] 
                for f in facilities]
    low_util = [utilization_df[(utilization_df['Facility'] == f) & (utilization_df['Scenario'] == 'Low Demand')]['Utilization'].values[0] 
               for f in facilities]
    
    ax3.bar(x - width/2, high_util, width, label='High Demand', color='#FF6B6B', alpha=0.8)
    ax3.bar(x + width/2, low_util, width, label='Low Demand', color='#4ECDC4', alpha=0.8)
    
    ax3.set_xlabel('Facility')
    ax3.set_ylabel('Capacity Utilization (%)')
    ax3.set_title('Capacity Utilization by Scenario')
    ax3.set_xticks(x)
    ax3.set_xticklabels(facilities)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Scenario cost comparison
    ax4 = axes[1, 1]
    scenario_names = ['High Demand', 'Low Demand']
    scenario_costs = [solution['variable_costs'][s] for s in instance.scenarios]
    
    bars = ax4.bar(scenario_names, scenario_costs, color=['#FF6B6B', '#4ECDC4'])
    ax4.set_ylabel('Variable Cost ($)')
    ax4.set_title('Variable Cost by Scenario')
    ax4.grid(True, alpha=0.3)
    
    # Add value labels and probabilities
    for bar, cost, prob in zip(bars, scenario_costs, instance.scenario_probs):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(scenario_costs)*0.01,
                f'${cost:,.0f}\n(p={prob})', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Visualize the solution
if solution:
    visualize_network_solution(instance, solution)

In [ ]:
def sensitivity_analysis(instance):
    """Perform sensitivity analysis on key parameters"""
    
    print("\n=== SENSITIVITY ANALYSIS ===")
    
    # 1. Vary demand levels
    print("\n1. Demand Sensitivity Analysis:")
    demand_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
    
    demand_results = []
    for multiplier in demand_multipliers:
        # Create modified instance
        modified_instance = create_example_instance()
        
        # Scale demands
        for key in modified_instance.demands:
            modified_instance.demands[key] *= multiplier
        
        # Solve
        prob, y_vars, x_vars, z_vars = solve_supply_chain_network_design(modified_instance)
        sol = extract_solution(modified_instance, prob, y_vars, x_vars, z_vars)
        
        if sol:
            demand_results.append({
                'Demand_Multiplier': multiplier,
                'Total_Cost': sol['total_cost'],
                'Opened_Facilities': len(sol['opened_facilities'])
            })
    
    demand_df = pd.DataFrame(demand_results)
    print(demand_df.to_string(index=False))
    
    # 2. Vary capacity levels
    print("\n\n2. Capacity Sensitivity Analysis:")
    capacity_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
    
    capacity_results = []
    for multiplier in capacity_multipliers:
        # Create modified instance
        modified_instance = create_example_instance()
        
        # Scale capacities
        modified_instance.capacities = [cap * multiplier for cap in modified_instance.capacities]
        
        # Solve
        prob, y_vars, x_vars, z_vars = solve_supply_chain_network_design(modified_instance)
        sol = extract_solution(modified_instance, prob, y_vars, x_vars, z_vars)
        
        if sol:
            capacity_results.append({
                'Capacity_Multiplier': multiplier,
                'Total_Cost': sol['total_cost'],
                'Opened_Facilities': len(sol['opened_facilities'])
            })
    
    capacity_df = pd.DataFrame(capacity_results)
    print(capacity_df.to_string(index=False))
    
    # Create sensitivity plots
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Demand sensitivity
    ax1 = axes[0]
    ax1.plot(demand_df['Demand_Multiplier'], demand_df['Total_Cost'], 'o-', linewidth=2, markersize=8, color='#FF6B6B')
    ax1.set_xlabel('Demand Multiplier')
    ax1.set_ylabel('Total Expected Cost ($)')
    ax1.set_title('Demand Sensitivity Analysis')
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(demand_multipliers)
    
    # Capacity sensitivity
    ax2 = axes[1]
    ax2.plot(capacity_df['Capacity_Multiplier'], capacity_df['Total_Cost'], 'o-', linewidth=2, markersize=8, color='#4ECDC4')
    ax2.set_xlabel('Capacity Multiplier')
    ax2.set_ylabel('Total Expected Cost ($)')
    ax2.set_title('Capacity Sensitivity Analysis')
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(capacity_multipliers)
    
    plt.tight_layout()
    plt.show()

# Perform sensitivity analysis
sensitivity_analysis(instance)

### Key Insights from Mathematical Formulation

The integer programming formulation provides several important insights:

1. **Optimal Facility Selection**: The model identifies which facilities should be opened based on the trade-off between fixed costs and expected variable costs.

2. **Scenario-Based Assignment**: Customer assignments can vary by scenario, providing flexibility to handle demand uncertainty.

3. **Capacity Utilization**: The solution shows how capacity is utilized differently across demand scenarios.

4. **Cost Structure**: Clear breakdown between fixed facility costs and expected variable operational costs.

### Mathematical Model Strengths

- **Optimality Guarantee**: Provides provably optimal solutions
- **Uncertainty Handling**: Explicitly models multiple scenarios
- **Comprehensive Constraints**: Handles capacity, assignment, and logical constraints
- **Transparency**: Clear mathematical formulation and solution interpretation

### Limitations and Computational Considerations

- **Scalability**: Computational complexity grows exponentially with problem size
- **Scenario Limitation**: Requires discrete scenarios, not continuous distributions
- **Data Requirements**: Needs accurate cost and demand data for all scenarios

This mathematical foundation serves as the benchmark against which all other approaches will be compared in subsequent tiers.